# Stage C v1 in Colab (3 steps)

1. **Runtime → Change runtime type → T4 GPU**
2. **Run Cell 1** – install deps (transformers 4.44.2 goes to a local folder). **Do not restart runtime** after Cell 1.
3. **Run Cell 2** – upload your ZIP, then upload **both** `stage_b_text_chunks.json` and `stage_b_table_triples.json` (saved to `outputs/STAGE_B_v1/`)
4. **Run Cell 3** – run Stage C v1 (output: `outputs/STAGE_C_v1/stage_c_statements_with_entities.json` with statements + table_triples)

In [ ]:
# Cell 1: Install transformers 4.44.2 into a LOCAL folder (so Colab's preinstalled version is not used)
# Do NOT install docling here — Stage C only needs NER. Do NOT restart runtime after this cell.
import os
os.makedirs("/content/transformers_444", exist_ok=True)
!pip install -q "torch>=2.0.0" pymupdf pdfplumber
!pip install -q --target /content/transformers_444 "transformers==4.44.2"
import sys
sys.path.insert(0, "/content/transformers_444")
import transformers
print("transformers version (from /content/transformers_444):", transformers.__version__)

In [ ]:
# Cell 2: Upload project ZIP, then upload both Stage B JSON files
import sys
import shutil
from google.colab import files
import zipfile
from pathlib import Path

print("Upload your Thesis_colab.zip (or project zip)...")
uploaded = files.upload()
zip_name = list(uploaded.keys())[0]
with zipfile.ZipFile(zip_name, 'r') as z:
    z.extractall("/content")
print("ZIP extracted.")

ROOT = Path("/content/Thesis") if (Path("/content") / "Thesis").exists() else Path("/content")
sys.path.insert(0, str(ROOT))
print(f"Project root: {ROOT}")

# outputs/STAGE_B_v1 for Stage C v1 input (align with local)
STAGE_B_V1_DIR = Path("/content/outputs/STAGE_B_v1")
STAGE_B_V1_DIR.mkdir(parents=True, exist_ok=True)

print("\nNow upload BOTH stage_b_text_chunks.json AND stage_b_table_triples.json (you can select both)...")
up2 = files.upload()
for fname in up2:
    if "stage_b" in fname and fname.endswith(".json"):
        dest = STAGE_B_V1_DIR / fname
        shutil.move(fname, dest)
        print(f"  Saved: {dest}")
print(f"Stage B v1 files ready in: {STAGE_B_V1_DIR}")

In [ ]:
# Cell 3: Run Stage C v1 (load both Stage B outputs, run NER, save statements + table_triples)
# Load our transformers 4.44.2 first (must be before any pipeline import)
import sys
sys.path.insert(0, "/content/transformers_444")

import json
from pathlib import Path
from google.colab import files

ROOT = Path("/content/Thesis") if Path("/content/Thesis").exists() else Path("/content")
sys.path.insert(0, str(ROOT))

STAGE_B_V1_DIR = Path("/content/outputs/STAGE_B_v1")
CHUNKS_PATH = STAGE_B_V1_DIR / "stage_b_text_chunks.json"
TRIPLES_PATH = STAGE_B_V1_DIR / "stage_b_table_triples.json"
if not CHUNKS_PATH.exists():
    raise FileNotFoundError("stage_b_text_chunks.json not found. Run Cell 2 and upload both Stage B JSON files.")

from pipeline.data import TextChunks, StatementsWithMedicalEntities
from pipeline.models import NeuralModel
from pipeline.inference import RecognizeEntities

# Load text chunks
with open(CHUNKS_PATH, "r", encoding="utf-8") as f:
    chunks_data = json.load(f)
chunks = TextChunks()
for c in chunks_data["chunks"]:
    chunks.add_chunk(page=c["page"], text=c["text"], source=c.get("source", ""), chunk_id=c.get("chunk_id"))

# Load table triples
table_triples = []
if TRIPLES_PATH.exists():
    with open(TRIPLES_PATH, "r", encoding="utf-8") as f:
        triples_data = json.load(f)
    table_triples = triples_data.get("triples", []) if isinstance(triples_data, dict) else triples_data

# Run NER
model = NeuralModel(model_name="d4data/biomedical-ner-all")
recognizer = RecognizeEntities(neural_model=model, min_score=0.55, acronym_file=None)
result = recognizer.infer(chunks)

table_triples = recognizer.enrich_triples_with_entities(table_triples)

# Write to outputs/STAGE_C_v1 (statements + table_triples)
STAGE_C_V1_DIR = Path("/content/outputs/STAGE_C_v1")
STAGE_C_V1_DIR.mkdir(parents=True, exist_ok=True)
out_path = STAGE_C_V1_DIR / "stage_c_statements_with_entities.json"
out_data = {
    "metadata": {
        "stage": "c",
        "description": "Statements with medical entities (NER + acronym expansion) and table triples from Stage B",
        "total_statements": result.count(),
        "total_entities": result.get_entity_count(),
        "total_table_triples": len(table_triples),
        "neural_model": "d4data/biomedical-ner-all",
        "min_ner_score": 0.55,
    },
    "statements": result.get_all(),
    "table_triples": table_triples,
}
out_path.write_text(json.dumps(out_data, indent=2, ensure_ascii=False), encoding="utf-8")
print(f"Done. Statements: {result.count()}, Entities: {result.get_entity_count()}, Table triples: {len(table_triples)}")
print(f"Output: {out_path}")
files.download(str(out_path))

STAGE_B_V1_DIR = Path("/content/outputs/STAGE_B_v1")
CHUNKS_PATH = STAGE_B_V1_DIR / "stage_b_text_chunks.json"
TRIPLES_PATH = STAGE_B_V1_DIR / "stage_b_table_triples.json"
if not CHUNKS_PATH.exists():
    raise FileNotFoundError("stage_b_text_chunks.json not found. Run Cell 2 and upload both Stage B JSON files.")

from pipeline.data import TextChunks, StatementsWithMedicalEntities
from pipeline.models import NeuralModel
from pipeline.inference import RecognizeEntities

# Load text chunks
with open(CHUNKS_PATH, "r", encoding="utf-8") as f:
    chunks_data = json.load(f)
chunks = TextChunks()
for c in chunks_data["chunks"]:
    chunks.add_chunk(page=c["page"], text=c["text"], source=c.get("source", ""), chunk_id=c.get("chunk_id"))

# Load table triples
table_triples = []
if TRIPLES_PATH.exists():
    with open(TRIPLES_PATH, "r", encoding="utf-8") as f:
        triples_data = json.load(f)
    table_triples = triples_data.get("triples", []) if isinstance(triples_data, dict) else triples_data

# Run NER
model = NeuralModel(model_name="d4data/biomedical-ner-all")
recognizer = RecognizeEntities(neural_model=model, min_score=0.55, acronym_file=None)
result = recognizer.infer(chunks)

table_triples = recognizer.enrich_triples_with_entities(table_triples)

# Write to outputs/STAGE_C_v1 (statements + table_triples)
STAGE_C_V1_DIR = Path("/content/outputs/STAGE_C_v1")
STAGE_C_V1_DIR.mkdir(parents=True, exist_ok=True)
out_path = STAGE_C_V1_DIR / "stage_c_statements_with_entities.json"
out_data = {
    "metadata": {
        "stage": "c",
        "description": "Statements with medical entities (NER + acronym expansion) and table triples from Stage B",
        "total_statements": result.count(),
        "total_entities": result.get_entity_count(),
        "total_table_triples": len(table_triples),
        "neural_model": "d4data/biomedical-ner-all",
        "min_ner_score": 0.55,
    },
    "statements": result.get_all(),
    "table_triples": table_triples,
}
out_path.write_text(json.dumps(out_data, indent=2, ensure_ascii=False), encoding="utf-8")
print(f"Done. Statements: {result.count()}, Entities: {result.get_entity_count()}, Table triples: {len(table_triples)}")
print(f"Output: {out_path}")
files.download(str(out_path))